### Step 3d: Two objectives and the trade-off curve

Same model as Step 3c. What's new: **two goals that conflict.**

1. **Maximize expected revenue.**
2. **Minimize expected makegood cost.**

They conflict because every sale adds revenue, but every guarantee above the baseline also adds makegood risk. You can't have the most revenue *and* the least risk. Step 3c combined them into one number (profit = revenue − makegood cost). This step keeps them separate and shows the whole range of choices.

**Method: a makegood budget (course Module 3).**
- Add one constraint: expected makegood cost ≤ budget.
- Maximize revenue for a range of budgets, from \$0 (no risk allowed) up to the riskiest plan.
- Each budget gives one point: the most revenue possible for that much risk. Connecting the points gives the **trade-off curve**.
- Ed's profit-maximizing plan (Step 3c) is one point on the curve.

**What changed in the code (sections 11-13 only; sections 1-10 are the same as Step 3c):**
- Section 11 defines the two objectives, `total_revenue` and `total_penalty`, plus **Constraint 4**, the makegood budget.
- Section 12 solves the model several times:
  - **A.** Maximize revenue with no budget, which finds the riskiest end of the curve.
  - **B.** Maximize profit (Ed's view).
  - **C.** For each budget level, set the budget and maximize revenue. `budget.RHS = level` changes the constraint's right-hand side: same model, new limit.
- Section 13 prints a table of every point and draws the curve.
- New import: `matplotlib.pyplot` for the chart.

**Size (rubric):** 360 decision variables, 392 constraints in 4 families (one package per customer, CPM requirement, 20 spots, makegood budget), and 2 conflicting objectives.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
import course_utils  # course helpers (print_status, print_model, ...)

In [ ]:
# ============================================================
# LOAD THE DATA
# ============================================================

# Ed's spreadsheet. Keep it in the same folder as this notebook.
# We only read from it; we never edit it.

DATA_FILE = "NFL Ad PKG MAIN.xlsx"


# ------------------------------------------------------------
# A. The package: 'Pre-Schedule Release', rows 13-17
# ------------------------------------------------------------

# One row per game type (Thanksgiving, noon regional, ...).
# This is what the station knows in July, before the schedule is out.
#
# col C = game type
# col E = number of games
# col F = rate per spot ($)
# col H = audience per game (000s = thousands of households)

df_games = pd.read_excel(
    DATA_FILE,
    sheet_name="Pre-Schedule Release",
    header=None,
    skiprows=12,
    nrows=5,
    usecols="C,E,F,H"
)

df_games.columns = ["game_type", "num_games", "rate", "audience_000s"]

package_price = (df_games["num_games"] * df_games["rate"]).sum()
baseline_000s = (df_games["num_games"] * df_games["audience_000s"]).sum()

# Check against the spreadsheet's own totals (row 18)
assert df_games["num_games"].sum() == 27
assert package_price == 108_000
assert abs(baseline_000s - 1631.3) < 0.001


# ------------------------------------------------------------
# B. Ratings scenarios: 'Schedule Release 1-Min Qualifer', T6:U8
# ------------------------------------------------------------

# Nielsen's new 1-minute rule raises ratings, but nobody knows
# by how much. Management gives 3 possible increases.
#
# col T = label, such as "15% Increase"
# col U = probability of that increase

df_scen = pd.read_excel(
    DATA_FILE,
    sheet_name="Schedule Release 1-Min Qualifer",
    header=None,
    skiprows=5,
    nrows=3,
    usecols="T:U"
)

df_scen.columns = ["label", "probability"]

# "15% Increase" -> 0.15
df_scen["increase"] = df_scen["label"].str.split("%").str[0].astype(float) / 100

# Check: probabilities add up to 100%
assert abs(df_scen["probability"].sum() - 1.0) < 0.000001


# ------------------------------------------------------------
# C. Prospects: 'Prospect Data Table', rows 4-33
# ------------------------------------------------------------

# col B = prospect ID
# col C = prospect name
# col D = salesperson        (not used until a later step)
# col E = close ratio        (not used until a later step)
# col F = goal CPM
# col G = makegood penalty multiplier (2, 3, or 4)

df_p = pd.read_excel(
    DATA_FILE,
    sheet_name="Prospect Data Table",
    skiprows=2,
    nrows=30,
    usecols="B:G"
)

df_p.columns = ["id", "prospect", "salesperson", "close_ratio",
                "goal_cpm", "penalty"]

# Check: 30 prospects, 6 salespeople with 5 prospects each
assert len(df_p) == 30
assert df_p["salesperson"].value_counts().tolist() == [5] * 6


# ------------------------------------------------------------
# D. The real schedule: 'Schedule Release 1-Min Qualifer', rows 13-39
# ------------------------------------------------------------

# One row per actual game, released in August. Games with
# low-interest teams are already cut 30% (one team) or 40% (two).
#
# col C = game ID
# col D = game name
# col F = rate per spot ($)
# col H = original audience per game (000s)
# col I = revised audience after the dud-game cuts (000s)

df_sched = pd.read_excel(
    DATA_FILE,
    sheet_name="Schedule Release 1-Min Qualifer",
    header=None,
    skiprows=12,
    nrows=27,
    usecols="C,D,F,H,I"
)

df_sched.columns = ["game_id", "game", "rate", "original_000s", "revised_000s"]

revised_000s = df_sched["revised_000s"].sum()

# Check against the spreadsheet's own totals (row 40)
assert len(df_sched) == 27
assert df_sched["rate"].sum() == 108_000
assert abs(df_sched["original_000s"].sum() - 1651.38) < 0.001
assert abs(revised_000s - 1465.938) < 0.001


print(f"Package price:  ${package_price:,.0f}")
print(f"Baseline 000s:  {baseline_000s:,.1f}   (July: generic games)")
print(f"Revised 000s:   {revised_000s:,.1f}   (August: dud games cut)")
print()
print(df_scen)
print()
df_p

In [ ]:
# ============================================================
# 1. INDEX SETS
# ============================================================

# customers: the 30 prospects, read from the spreadsheet

customers = df_p["id"].tolist()

# packages: how much audience we guarantee, above the baseline.
# Same 4 names as the base model. The levels match the
# 3 possible Nielsen increases, plus no increase at all.

packages = [
    "conservative",
    "medium",
    "premium",
    "aggressive"
]

# discounts: how much we take off the $108,000 price (NEW)

discounts = [0.00, 0.05, 0.10]

# scenarios: the 3 possible Nielsen increases, from the spreadsheet

scenarios = df_scen["label"].tolist()


# ============================================================
# 2. CUSTOMER PARAMETERS
# ============================================================

goal_cpm = dict(zip(df_p["id"], df_p["goal_cpm"]))

# Makegood multiplier: if we fall short, customer c is owed
# penalty[c] times the value of the missing audience.

underdelivery_penalty = dict(zip(df_p["id"], df_p["penalty"]))


# ============================================================
# 3. AD PACKAGE PARAMETERS
# ============================================================

# Guarantee level for each package (TEAM ASSUMPTION)

guarantee = {
    "conservative": 0.00,
    "medium": 0.15,
    "premium": 0.20,
    "aggressive": 0.25
}

# Audience we promise the customer (000s)

guaranteed_000s = {}

for p in packages:
    guaranteed_000s[p] = baseline_000s * (1 + guarantee[p])

# Price and CPM now depend on the package AND the discount.
# A bigger guarantee or a bigger discount means a lower CPM.

price = {}
cpm = {}

for p in packages:
    for d in discounts:
        price[p, d] = package_price * (1 - d)
        cpm[p, d] = price[p, d] / guaranteed_000s[p]


# ============================================================
# 4. ACTUAL AUDIENCE OUTCOMES
# ============================================================

# 3 separate outcomes that already add up to 100%.
#
# The station plans for the dud games: the audience it will
# actually deliver starts from the REVISED audience (after the
# dud-game cuts), not the July baseline.
#
# Actual audience = revised x (1 + Nielsen increase)
#
# (Guarantees in section 3 stay on the July baseline, because
#  that's what the customer was promised.)

probability = dict(zip(df_scen["label"], df_scen["probability"]))

actual_000s = {}

for s in scenarios:
    increase = df_scen.loc[df_scen["label"] == s, "increase"].iloc[0]
    actual_000s[s] = revised_000s * (1 + increase)


# ============================================================
# 5. DETERMINE PACKAGE ELIGIBILITY
# ============================================================

# eligible[c,p,d] = 1 if customer c's goal CPM is high enough
# to accept package p at discount d.
#
# Otherwise eligible[c,p,d] = 0.

eligible = {}

for c in customers:
    for p in packages:
        for d in discounts:

            if cpm[p, d] <= goal_cpm[c]:
                eligible[c, p, d] = 1
            else:
                eligible[c, p, d] = 0


# ============================================================
# 6. CALCULATE EXPECTED UNDERDELIVERY PENALTY
# ============================================================

# Shortfall (000s) x CPM the customer paid ($ per 000) = value
# of the missing audience in dollars. The customer is owed that
# value times their penalty multiplier.

expected_penalty = {}

for c in customers:
    for p in packages:
        for d in discounts:

            expected_penalty[c, p, d] = sum(
                probability[s] *
                max(guaranteed_000s[p] - actual_000s[s], 0) *
                cpm[p, d] *
                underdelivery_penalty[c]
                for s in scenarios
            )


# ============================================================
# 7. CALCULATE EXPECTED PROFIT
# ============================================================

expected_profit = {}

for c in customers:
    for p in packages:
        for d in discounts:

            expected_profit[c, p, d] = (
                price[p, d]
                - expected_penalty[c, p, d]
            )


# ============================================================
# 8. CREATE GUROBI MODEL
# ============================================================

model = gp.Model("NFL_Advertising_Tradeoff")

model.Params.OutputFlag = 0   # we solve many times, so hide the solver log


# ============================================================
# 9. DECISION VARIABLES
# ============================================================

# x[c,p,d] = 1 if customer c receives package p at discount d
#            0 otherwise

x = model.addVars(
    customers,
    packages,
    discounts,
    vtype=GRB.BINARY,
    name="x"
)


# ============================================================
# 10. CONSTRAINTS
# ============================================================

# ------------------------------------------------------------
# Constraint 1:
# Each customer can receive at most one package
# (one guarantee level at one discount).
# ------------------------------------------------------------

model.addConstrs(
    (
        gp.quicksum(x[c, p, d] for p in packages for d in discounts) <= 1
        for c in customers
    ),
    name="one_package_per_customer"
)


# ------------------------------------------------------------
# Constraint 2:
# Package CPM cannot exceed customer's goal CPM.
# ------------------------------------------------------------

model.addConstrs(
    (
        x[c, p, d] <= eligible[c, p, d]
        for c in customers
        for p in packages
        for d in discounts
    ),
    name="cpm_requirement"
)


# ------------------------------------------------------------
# Constraint 3:
# We have 20 advertising spots, so we can sell at most 20 packages.
# ------------------------------------------------------------

model.addConstr(
    gp.quicksum(
        x[c, p, d]
        for c in customers
        for p in packages
        for d in discounts
    ) <= 20,
    name="advertising_spots"
)


# ============================================================
# 11. TWO OBJECTIVES (NEW)
# ============================================================

# Objective 1: maximize expected revenue

total_revenue = gp.quicksum(
    price[p, d] * x[c, p, d]
    for c in customers
    for p in packages
    for d in discounts
)

# Objective 2: minimize expected makegood cost

total_penalty = gp.quicksum(
    expected_penalty[c, p, d] * x[c, p, d]
    for c in customers
    for p in packages
    for d in discounts
)

# These conflict: every sale adds revenue, but every guarantee
# above the baseline also adds makegood risk.


# ------------------------------------------------------------
# Constraint 4 (NEW):
# Makegood budget. Expected makegood cost cannot exceed the budget.
# We start with no limit, and change the budget in section 12.
# ------------------------------------------------------------

budget = model.addConstr(
    total_penalty <= GRB.INFINITY,
    name="makegood_budget"
)


# ============================================================
# 12. SOLVE: TRACE THE TRADE-OFF CURVE
# ============================================================

# Step A: maximize revenue with no limit on risk.
#         This is the riskiest end of the curve.

model.setObjective(total_revenue, GRB.MAXIMIZE)
model.optimize()

max_penalty = total_penalty.getValue()


# Step B: Ed's view. Maximize expected profit (revenue minus
#         makegood cost), with no budget. This is the Step 3c
#         answer, and one point on the curve.

model.setObjective(total_revenue - total_penalty, GRB.MAXIMIZE)
model.optimize()

best_revenue = total_revenue.getValue()
best_penalty = total_penalty.getValue()


# Step C: set the budget to 10 evenly spaced levels between
#         0 (no risk allowed) and max_penalty, plus Ed's point.
#         For each budget, maximize revenue again.

num_points = 10

budget_levels = [max_penalty * i / num_points for i in range(num_points + 1)]
budget_levels.append(best_penalty)
budget_levels.sort()

model.setObjective(total_revenue, GRB.MAXIMIZE)

results = []

for level in budget_levels:

    budget.RHS = level

    model.optimize()

    revenue = total_revenue.getValue()
    penalty = total_penalty.getValue()

    results.append({
        "budget": level,
        "revenue": revenue,
        "penalty": penalty,
        "profit": revenue - penalty,
        "sold": sum(1 for k in x if x[k].X > 0.5),
        "conservative": sum(1 for (c, p, d) in x if p == "conservative" and x[c, p, d].X > 0.5),
        "medium": sum(1 for (c, p, d) in x if p == "medium" and x[c, p, d].X > 0.5),
        "premium": sum(1 for (c, p, d) in x if p == "premium" and x[c, p, d].X > 0.5),
        "aggressive": sum(1 for (c, p, d) in x if p == "aggressive" and x[c, p, d].X > 0.5)
    })

df_results = pd.DataFrame(results)


# ============================================================
# 13. DISPLAY RESULTS
# ============================================================

print("TRADE-OFF: best revenue for each makegood budget")
print("=" * 75)
print(
    df_results.to_string(
        index=False,
        formatters={
            "budget": "${:,.0f}".format,
            "revenue": "${:,.0f}".format,
            "penalty": "${:,.0f}".format,
            "profit": "${:,.0f}".format
        }
    )
)

print()
print("Ed's view (maximize revenue minus makegood cost):")
print(f"  Revenue ${best_revenue:,.0f} | Makegood cost ${best_penalty:,.0f} | "
      f"Profit ${best_revenue - best_penalty:,.0f}")


# ------------------------------------------------------------
# Chart: revenue (vertical) against makegood cost (horizontal)
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    df_results["penalty"] / 1000,
    df_results["revenue"] / 1000,
    marker="o", markersize=8, linewidth=2,
    color="#2a78d6"
)

ax.plot(
    best_penalty / 1000,
    best_revenue / 1000,
    marker="o", markersize=12, linestyle="none",
    color="#eb6834", markeredgecolor="white", markeredgewidth=2
)

ax.annotate(
    f"Max profit (Ed's view)\n${(best_revenue - best_penalty) / 1000:,.0f}K profit",
    xy=(best_penalty / 1000, best_revenue / 1000),
    xytext=(25, -70), textcoords="offset points",
    color="#0b0b0b", fontsize=10,
    arrowprops=dict(arrowstyle="-", color="#52514e")
)

ax.set_title("Revenue vs. makegood risk", loc="left", fontsize=13, color="#0b0b0b")
ax.set_xlabel("Expected makegood cost ($ thousands)", color="#52514e")
ax.set_ylabel("Expected revenue ($ thousands)", color="#52514e")
ax.grid(color="#e6e5e0", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

### What the results show

| Makegood budget | Revenue | Makegood cost | Profit | Sold | Medium | Premium | Aggressive |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| \$0 | \$0 | \$0 | \$0 | 0 | 0 | 0 | 0 |
| \$70,690 | \$324,000 | \$65,751 | \$258,249 | 3 | 2 | 1 | 0 |
| \$141,380 | \$685,800 | \$141,226 | \$544,574 | 7 | 6 | 1 | 0 |
| \$212,070 | \$918,000 | \$208,396 | \$709,604 | 9 | 8 | 0 | 1 |
| \$282,761 | \$1,204,200 | \$280,723 | \$923,477 | 12 | 10 | 1 | 1 |
| \$353,451 | \$1,377,000 | \$352,889 | \$1,024,111 | 14 | 10 | 2 | 2 |
| \$424,141 | \$1,506,600 | \$419,163 | \$1,087,437 | 15 | 9 | 2 | 4 |
| \$494,831 | \$1,614,600 | \$494,385 | \$1,120,215 | 16 | 8 | 3 | 5 |
| **\$504,307 (Ed's point)** | **\$1,690,200** | **\$504,307** | **\$1,185,893** | **17** | **10** | **3** | **4** |
| \$565,521 | \$1,717,200 | \$561,193 | \$1,156,007 | 17 | 6 | 6 | 5 |
| \$636,211 | \$1,744,200 | \$631,900 | \$1,112,300 | 17 | 5 | 3 | 9 |
| \$706,902 | \$1,760,400 | \$706,902 | \$1,053,498 | 17 | 2 | 4 | 11 |

No customer ever gets conservative: its CPM is above every prospect's goal.

**How to read the curve:** its steepness is the revenue gained for each extra dollar of makegood risk.

- **Left end (steep): risk pays well.** The first \$140K of risk buys about \$686K of revenue, roughly \$5 per \$1 of risk. The model sells mostly **medium**, the safest guarantee customers will buy.
- **Middle: returns shrink.** Each extra budget step still adds customers, but they need bigger guarantees, so each dollar of risk buys less revenue.
- **Ed's point (\$504K of risk): the best profit.** This is where extra risk stops paying for itself. Up to here, each extra dollar of risk brings in more than a dollar of revenue.
- **Right end (flat): risk stops paying.** Going from Ed's point to the riskiest plan adds \$203K of risk for only \$70K more revenue. All 17 customers are already sold. The extra revenue comes only from moving customers to **aggressive** at smaller discounts, which is why profit falls.

**This is where discounts matter.** Moving left from the right end, customers shift from aggressive (11 → 4) to medium (2 → 10): from "big guarantee, full price" to "safe guarantee, a bit off." Without discounts, the only way to cut risk would be to drop customers.

**Why the curve isn't perfectly smooth:** each sale is all or nothing (a yes/no variable), so the best plan can jump between budgets. For example, going from a \$494,831 budget to \$504,307 adds only \$9,922 of risk but \$75,600 of revenue: one more customer fits, and several customers switch options. That's also why we add Ed's point to the list of budgets, so the curve passes through it.

**Choosing a point is a management decision.** Ed's point gives the highest expected profit. A more cautious station could choose a point to the left: less revenue, but less exposure if ratings disappoint. The curve shows exactly what each choice costs.

### Changes from Step 3c

1. **Section 11:** the objective is split into `total_revenue` and `total_penalty`, and a new **Constraint 4** caps the makegood cost at a budget.
2. **Section 12:** the model is solved several times: riskiest plan, Ed's profit point, then revenue at each budget level.
3. **Section 13:** a table of every point and the trade-off chart (also saved as `Group7_Step3d_Tradeoff.png`).
4. `model.Params.OutputFlag = 0` hides the solver log, since we solve 14 times.
5. The data cell and sections 1-10 are unchanged from Step 3c.